Perfect 👍 Let’s upgrade your **LangGraph project** so that the **router agent** uses an **LLM** (OpenAI/Gemini) instead of just rules.
This way, the **parent router** can understand free-form prompts and decide which child agent (weather/pollution) should handle it.

---

# 📂 Final Project Scaffolding

```
order_mgmt_agents/
│── main.py
│── config/
│   ├── __init__.py
│   ├── settings.py
│── tools/
│   ├── __init__.py
│   ├── weather_tools.py
│   ├── pollution_tools.py
│── mcp/
│   ├── __init__.py
│   ├── server1.py
│   ├── server2.py
│   ├── server3.py
│   ├── server4.py
│── agents/
│   ├── __init__.py
│   ├── agent_factory.py
│   ├── weather_agent.py
│   ├── pollution_agent.py
│   ├── parent_agent.py
```

---

# ⚙️ `config/settings.py`

```python
import os
from dotenv import load_dotenv

# Load environment variables
dotenv_path = os.path.join(os.path.dirname(__file__), "../.env")
load_dotenv(dotenv_path)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "fake-key")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "fake-gemini-key")

# Agent Config
AGENT_CONFIG = {
    "weather": {
        "llm": "openai",
        "mcp_servers": ["server1", "server2"],
        "tools": ["get_city_weather", "get_country_weather"]
    },
    "pollution": {
        "llm": "gemini",
        "mcp_servers": ["server3", "server4"],
        "tools": ["get_city_pollution", "get_country_pollution"]
    },
}
```

---

# 🛠️ `tools/weather_tools.py`

```python
import requests

def get_city_weather(city: str) -> str:
    """Fetch weather for a given city."""
    try:
        url = f"https://wttr.in/{city}?format=3"
        return requests.get(url, timeout=5).text
    except Exception as e:
        return f"Error fetching weather: {e}"

def get_country_weather(country: str) -> str:
    return f"Average weather data for {country} (mocked)"
```

---

# 🛠️ `tools/pollution_tools.py`

```python
def get_city_pollution(city: str) -> str:
    return f"Pollution level in {city}: Moderate (mocked)"

def get_country_pollution(country: str) -> str:
    return f"Average pollution in {country}: High (mocked)"
```

---

# 🤖 `agents/weather_agent.py`

```python
from tools.weather_tools import get_city_weather, get_country_weather

def create_weather_agent():
    def run(task: dict):
        city = task.get("city")
        country = task.get("country")

        if city:
            return get_city_weather(city)
        if country:
            return get_country_weather(country)

        return "Weather agent: No location provided"
    return run
```

---

# 🤖 `agents/pollution_agent.py`

```python
from tools.pollution_tools import get_city_pollution, get_country_pollution

def create_pollution_agent():
    def run(task: dict):
        city = task.get("city")
        country = task.get("country")

        if city:
            return get_city_pollution(city)
        if country:
            return get_country_pollution(country)

        return "Pollution agent: No location provided"
    return run
```

---

# 🏭 `agents/agent_factory.py`

```python
from agents.weather_agent import create_weather_agent
from agents.pollution_agent import create_pollution_agent

class AgentFactory:
    def get_agent(self, agent_type: str):
        if agent_type == "weather":
            return create_weather_agent()
        elif agent_type == "pollution":
            return create_pollution_agent()
        else:
            raise ValueError(f"Unknown agent type: {agent_type}")
```

---

# 🤖 `agents/parent_agent.py` (LLM Router)

```python
from config.settings import OPENAI_API_KEY, GEMINI_API_KEY
from openai import OpenAI

# Initialize LLM client
client = OpenAI(api_key=OPENAI_API_KEY)

def llm_router(prompt: str) -> tuple:
    """
    Uses LLM to decide whether to route to weather_agent, pollution_agent, or end.
    Returns: (next_node, task_dict)
    """

    instruction = (
        "You are a router agent. Decide the correct agent.\n"
        "Options: weather_agent, pollution_agent, end\n"
        "Extract 'city' or 'country' if available.\n\n"
        f"User Prompt: {prompt}"
    )

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a smart router agent."},
            {"role": "user", "content": instruction}
        ],
        temperature=0
    )

    text = response.choices[0].message.content.strip().lower()

    # Simple parsing
    if "weather" in text:
        return "weather_agent", {"topic": "weather", "city": _extract_city(prompt), "country": _extract_country(prompt)}
    elif "pollution" in text:
        return "pollution_agent", {"topic": "pollution", "city": _extract_city(prompt), "country": _extract_country(prompt)}
    else:
        return "end", {"result": f"Parent agent: Unknown topic from '{prompt}'"}

def _extract_city(prompt: str):
    if "in" in prompt.lower():
        return prompt.split("in")[-1].strip()
    return None

def _extract_country(prompt: str):
    if "country" in prompt.lower():
        return prompt.split("country")[-1].strip()
    return None
```

---

# 🚀 `main.py`

```python
from langgraph.graph import StateGraph, START, END
from agents.parent_agent import llm_router
from agents.weather_agent import create_weather_agent
from agents.pollution_agent import create_pollution_agent

def build_graph():
    weather_agent = create_weather_agent()
    pollution_agent = create_pollution_agent()

    builder = StateGraph(dict)

    # Add nodes
    builder.add_node("parent_router", lambda state: llm_router(state["prompt"]))
    builder.add_node("weather_agent", lambda state: {"result": weather_agent(state[1])})
    builder.add_node("pollution_agent", lambda state: {"result": pollution_agent(state[1])})

    # Edges
    builder.add_edge(START, "parent_router")
    builder.add_conditional_edges(
        "parent_router",
        lambda output: output[0],  # LLM decides next node
        {"weather_agent": "weather_agent", "pollution_agent": "pollution_agent", "end": END}
    )
    builder.add_edge("weather_agent", END)
    builder.add_edge("pollution_agent", END)

    return builder.compile()

def main():
    graph = build_graph()

    prompts = [
        "What's the weather in London?",
        "Tell me the average weather in India",
        "Check pollution in Mumbai",
        "Give me pollution details for country India",
        "How is the traffic in Delhi?"
    ]

    for p in prompts:
        print(f"\n📝 Prompt: {p}")
        result = graph.invoke({"prompt": p})
        print("🤖 Response:", result.get("result"))

    drawable = graph.get_graph()
    print("\n=== Mermaid Graph ===")
    print(drawable.draw_mermaid())

    try:
        with open("agent_graph.png", "wb") as f:
            f.write(drawable.draw_mermaid_png())
        print("✅ Graph saved as agent_graph.png")
    except Exception as e:
        print("Could not save PNG:", e)

if __name__ == "__main__":
    main()
```

---

# 🔑 .env file

```
OPENAI_API_KEY=sk-your-openai-key
GEMINI_API_KEY=your-gemini-key
```

---

✅ Now the **router agent uses GPT-4o-mini** to classify and route tasks dynamically.
If the prompt is about *weather*, it triggers the **weather agent**; if it’s about *pollution*, it triggers the **pollution agent**; otherwise, it ends gracefully.

---

👉 Do you also want me to extend this so the **parent agent can choose between OpenAI and Gemini dynamically** (based on `AGENT_CONFIG`), instead of always using OpenAI?
